# Extended Thinking

## Table of contents
- [Setup](#setup)
- [Basic example](#basic-example)
- [Streaming with extended thinking](#streaming-with-extended-thinking)
- [Token counting and context window management](#token-counting-and-context-window-management)
- [Understanding redacted thinking](#understanding-redacted-thinking-blocks)
- [Handling error cases](#handling-error-cases)

This notebook demonstrates how to use Claude 3.7 Sonnet's extended thinking feature with various examples and edge cases.

Extended thinking gives Claude 3.7 Sonnet enhanced reasoning capabilities for complex tasks, while also providing transparency into its step-by-step thought process before it delivers its final answer. When extended thinking is turned on, Claude creates `thinking` content blocks where it outputs its internal reasoning. Claude incorporates insights from this reasoning before crafting a final response. For more information on extended thinking, see our [documentation](https://docs.claude.com/en/docs/build-with-claude/extended-thinking).

## Setup

First, let's install the necessary packages and set up our environment.

In [ ]:
// No pip install needed. Ensure dependencies are installed:
// npm install @anthropic-ai/sdk dotenv


In [ ]:
import Anthropic from '@anthropic-ai/sdk';

const client = new Anthropic();

function printThinkingResponse(response: Anthropic.Message): void {
  console.log('\n==== FULL RESPONSE ====');
  for (const block of response.content) {
    if (block.type === 'thinking') {
      console.log('\nTHINKING BLOCK:');
      const thinking = (block as any).thinking || '';
      console.log(thinking.length > 500 ? thinking.slice(0, 500) + '...' : thinking);
      const sig = (block as any).signature as string | undefined;
      console.log('\n[Signature available: ' + Boolean(sig) + ']');
      if (sig) console.log('[Signature (first 50 chars): ' + sig.slice(0, 50) + '...]');
    } else if (block.type === 'redacted_thinking') {
      console.log('\nREDACTED THINKING BLOCK:');
      const data = (block as any).data as string | undefined;
      console.log('[Data length: ' + (data ? data.length : 'N/A') + ']');
    } else if (block.type === 'text') {
      console.log('\nFINAL ANSWER:');
      console.log(block.text);
    }
  }
  console.log('\n==== END RESPONSE ====');
}

async function countTokens(messages: Anthropic.MessageParam[]): Promise<number> {
  const result = await client.messages.countTokens({
    model: 'claude-sonnet-4-6',
    messages,
  });
  return result.input_tokens;
}


## Basic example

Let's start with a basic example to show extended thinking in action:

In [ ]:
async function basicThinkingExample(): Promise<void> {
  const response = await client.messages.create({
    model: 'claude-sonnet-4-6',
    max_tokens: 4000,
    thinking: { type: 'enabled', budget_tokens: 2000 },
    messages: [{
      role: 'user',
      content: 'Solve this puzzle: Three people check into a hotel... Where is the missing $1?'
    }]
  });

  printThinkingResponse(response);
}

await basicThinkingExample();


## Streaming with extended thinking

This example shows how to handle streaming with thinking:

In [ ]:
async function streamingWithThinking(): Promise<void> {
  let currentBlockType: string | null = null;
  let currentContent = '';

  const stream = await client.messages.stream({
    model: 'claude-sonnet-4-6',
    max_tokens: 4000,
    thinking: { type: 'enabled', budget_tokens: 2000 },
    messages: [{
      role: 'user',
      content: 'Solve this puzzle: Three people check into a hotel... Where is the missing $1?'
    }]
  });

  for await (const event of stream) {
    if (event.type === 'content_block_start') {
      currentBlockType = event.content_block.type;
      console.log('\n--- Starting ' + currentBlockType + ' block ---');
      currentContent = '';
    } else if (event.type === 'content_block_delta') {
      if (event.delta.type === 'thinking_delta') {
        process.stdout.write(event.delta.thinking);
        currentContent += event.delta.thinking;
      } else if (event.delta.type === 'text_delta') {
        process.stdout.write(event.delta.text);
        currentContent += event.delta.text;
      }
    } else if (event.type === 'content_block_stop') {
      if (currentBlockType === 'thinking') {
        console.log('\n[Completed thinking block, ' + currentContent.length + ' characters]');
      }
      console.log('--- Finished ' + currentBlockType + ' block ---\n');
      currentBlockType = null;
    } else if (event.type === 'message_stop') {
      console.log('\n--- Message complete ---');
    }
  }
}

await streamingWithThinking();


## Token counting and context window management

This example demonstrates how to track token usage with extended thinking:

In [ ]:
async function tokenCountingExample(): Promise<void> {
  const baseMessages: Anthropic.MessageParam[] = [{
    role: 'user',
    content: 'Solve this puzzle: Three people check into a hotel... Where is the missing $1?'
  }];

  const baseTokenCount = await countTokens(baseMessages);
  console.log('Base token count (input only): ' + baseTokenCount);

  const response = await client.messages.create({
    model: 'claude-sonnet-4-6',
    max_tokens: 8000,
    thinking: { type: 'enabled', budget_tokens: 2000 },
    messages: baseMessages
  });

  const thinkingTokens = response.content
    .filter(b => b.type === 'thinking')
    .reduce((sum, b) => sum + (((b as any).thinking || '').split(' ').length * 1.3), 0);

  const finalAnswerTokens = response.content
    .filter(b => b.type === 'text')
    .reduce((sum, b) => sum + ((b as Anthropic.TextBlock).text.split(' ').length * 1.3), 0);

  console.log('Estimated thinking tokens used: ~' + Math.round(thinkingTokens));
  console.log('Estimated final answer tokens: ~' + Math.round(finalAnswerTokens));
  console.log('Total estimated output tokens: ~' + Math.round(thinkingTokens + finalAnswerTokens));

  const thinkingBudgets = [1024, 2000, 4000, 8000, 16000, 32000];
  const contextWindow = 200000;
  for (const budget of thinkingBudgets) {
    const needed = baseTokenCount + budget + 1000;
    console.log('\nWith thinking budget of ' + budget + ' tokens:');
    console.log('Max tokens needed: ' + needed);
    console.log('Remaining context window: ' + (contextWindow - needed));
    if (needed > contextWindow) console.log('WARNING: This would exceed the context window of 200k tokens!');
  }
}

await tokenCountingExample();


## Understanding redacted thinking blocks

Occasionally Claude's internal reasoning will be flagged by safety systems. When this occurs, we encrypt some or all of the `thinking` block and return it to you as a `redacted_thinking` block. These redacted thinking blocks are decrypted when passed back to the API, allowing Claude to continue its response without losing context.

This example demonstrates working with redacted thinking blocks using a special test string that triggers them:

In [ ]:
async function redactedThinkingExample(): Promise<void> {
  const response = await client.messages.create({
    model: 'claude-sonnet-4-6',
    max_tokens: 4000,
    thinking: { type: 'enabled', budget_tokens: 2000 },
    messages: [{
      role: 'user',
      content: 'ANTHROPIC_MAGIC_STRING_TRIGGER_REDACTED_THINKING_46C9A13E193C177646C7398A98432ECCCE4C1253D5E2D82641AC0E52CC2876CB'
    }]
  });

  const redactedBlocks = response.content.filter(b => b.type === 'redacted_thinking');
  const thinkingBlocks = response.content.filter(b => b.type === 'thinking');
  const textBlocks = response.content.filter(b => b.type === 'text') as Anthropic.TextBlock[];

  console.log('Response includes ' + response.content.length + ' total blocks:');
  console.log('- ' + redactedBlocks.length + ' redacted thinking blocks');
  console.log('- ' + thinkingBlocks.length + ' regular thinking blocks');
  console.log('- ' + textBlocks.length + ' text blocks');
}

await redactedThinkingExample();


## Handling error cases

When using extended thinking, keep in mind:

1. **Minimum budget**: The minimum thinking budget is 1,024 tokens. We suggest starting at the minimum and increasing incrementally to find the optimal range.

2. **Incompatible features**: Thinking isn't compatible with temperature, top_p, or top_k modifications, and you cannot pre-fill responses.

3. **Pricing**: Extended thinking tokens count towards the context window and are billed as output tokens. They also count towards your rate limits.

For more details on extended thinking with tool use, see the "Extended Thinking with Tool Use" notebook.

In [ ]:
async function demonstrateCommonErrors(): Promise<void> {
  try {
    await client.messages.create({
      model: 'claude-sonnet-4-6',
      max_tokens: 4000,
      thinking: { type: 'enabled', budget_tokens: 500 },
      messages: [{ role: 'user', content: 'Explain quantum computing.' }]
    });
  } catch (e) {
    console.log('Error with too small thinking budget: ' + e);
  }

  try {
    await client.messages.create({
      model: 'claude-sonnet-4-6',
      max_tokens: 4000,
      temperature: 0.7,
      thinking: { type: 'enabled', budget_tokens: 2000 },
      messages: [{ role: 'user', content: 'Write a creative story.' }]
    } as any);
  } catch (e) {
    console.log('Error with temperature and thinking: ' + e);
  }

  try {
    const longContent = 'Please analyze this text. ' + 'This is sample text. '.repeat(150000);
    await client.messages.create({
      model: 'claude-sonnet-4-6',
      max_tokens: 20000,
      thinking: { type: 'enabled', budget_tokens: 10000 },
      messages: [{ role: 'user', content: longContent }]
    });
  } catch (e) {
    console.log('Error from exceeding context window: ' + e);
  }
}

await demonstrateCommonErrors();
